# m42 lab · Physics-informed fitting of activation maps

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You fit activation maps to sparse, noisy activation times with and without a physics term (the eikonal residual),
see where physics helps and where a wrong physical assumption hides a feature, estimate uncertainty with an
ensemble, and build a fast surrogate of the eikonal model for calibration, checking where it is safe to use.

| Part | What you do | Data |
| --- | --- | --- |
| A | Radial-basis activation maps: analytic gradients and the eikonal residual (eq-m42-rbf, eq-m42-eikonal) | synthetic |
| B | Homogeneous tissue: data-only fit against physics-informed fit (eq-m42-loss, eq-m42-lm) | synthetic |
| C | A slow region: physics lowers map error but biases the speed estimated inside it | synthetic |
| D | Ensemble uncertainty: does the spread track the error? (eq-m42-ensemble) | synthetic |
| E | A surrogate of the eikonal model for calibration, inside and outside its training range (eq-m42-scaling) | synthetic |
| F | Optional: a physics-informed neural network in PyTorch | synthetic |

The fits use no deep-learning library: the activation map is a sum of Gaussian radial basis functions whose
gradients are analytic, fitted by Levenberg–Marquardt. Speeds (1.0 m/s, and 0.6 m/s in the slow region, as in m39,
from Heil et al. 2023), noise (1 ms) and geometry are illustrative. Each part ends with **checks**; the lab runs in
under a minute.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from scipy.interpolate import LinearNDInterpolator

from epml import mapping as mp
from epml import pinn

XG = pinn.grid_centres(0, 60, 31)        # evaluation grid, 2 mm
XC = pinn.grid_centres(0, 60, 21)        # collocation points for the physics term, 3 mm
CENT = pinn.grid_centres(0, 60, 9)       # 81 basis centres, 7.5 mm apart
H = 8.0                                  # basis width (mm), illustrative

def show(ax, f, title, pts=None, vmin=0, vmax=70, cmap="viridis"):
    im = ax.imshow(f.reshape(31, 31).T, origin="lower", extent=[-1, 61, -1, 61], vmin=vmin, vmax=vmax, cmap=cmap)
    if pts is not None:
        ax.plot(pts[:, 0], pts[:, 1], "w.", ms=5)
    ax.set(title=title, xticks=[0, 30, 60], yticks=[0, 30, 60])
    return im

## Part A · Maps made of radial basis functions

Check the analytic gradient of the basis against finite differences, and check that the exact activation map of
a point source, $T = \lvert \mathbf x - \mathbf x_0\rvert / v$, has zero eikonal residual with $s = 1/v$.

In [ ]:
rng = np.random.default_rng(42)
fit = pinn.ActivationFit(CENT, H, slowness=1.0)
fit.a = rng.normal(size=len(fit.a))
X = rng.uniform(5, 55, (20, 2))
eps = 1e-5
fd = np.stack([(fit.T(X + eps * e) - fit.T(X - eps * e)) / (2 * eps) for e in np.eye(2)], 1)
grad_err = np.abs(fd - fit.gradT(X)).max()

src, v = np.array([8.0, 30.0]), 1.0
d = XG - src
grad_exact = d / np.maximum(np.linalg.norm(d, axis=1, keepdims=True), 1e-12) / v
res = pinn.eikonal_residual(grad_exact[np.linalg.norm(d, axis=1) > 0], 1 / v)
print(f"largest gradient error {grad_err:.1e}; eikonal residual of the exact point-source map {np.abs(res).max():.1e}")

In [ ]:
# Checks
assert grad_err < 1e-6 and np.abs(res).max() < 1e-12

## Part B · Homogeneous tissue: does physics help?

Activation spreads from a point at (8, 30) mm at 1.0 m/s. Measure it at 12, 24 or 48 random points with 1 ms of
noise, then fit (i) data only ($\lambda = 0$) and (ii) data plus the eikonal term ($\lambda = 100$, illustrative)
with one unknown speed. Error is the root-mean-square difference from the true map over the whole 60-by-60-mm
square, averaged over six random draws.

In [ ]:
T_true = lambda X: np.linalg.norm(X - src, axis=1) / v
rows = {}
for seed in range(6):
    r = np.random.default_rng(100 + seed)
    for N in (12, 24, 48):
        Xd = r.uniform(0, 60, (N, 2))
        td = T_true(Xd) + r.normal(0, 1.0, N)
        for lam in (0, 100):
            f = pinn.ActivationFit(CENT, H, "scalar", s_prior=1.2)
            f.fit(Xd, td, XC, lam=lam, iters=80)
            rows.setdefault((N, lam), []).append((np.sqrt(np.mean((f.T(XG) - T_true(XG)) ** 2)), 1 / f.b[0]))
for (N, lam), vals in sorted(rows.items()):
    vals = np.array(vals)
    label = "data only" if lam == 0 else "physics  "
    print(f"N = {N:2d}  {label}  RMS error {vals[:, 0].mean():5.2f} ms" + ("" if lam == 0 else f"   speed estimate {vals[:, 1].mean():.2f} ± {vals[:, 1].std():.2f} m/s"))

r = np.random.default_rng(101)
Xd = r.uniform(0, 60, (24, 2)); td = T_true(Xd) + r.normal(0, 1.0, 24)
fig, axs = plt.subplots(1, 3, figsize=(10, 3.2))
show(axs[0], T_true(XG), "true map", Xd)
for ax, lam in zip(axs[1:], (0, 100)):
    f = pinn.ActivationFit(CENT, H, "scalar", s_prior=1.2); f.fit(Xd, td, XC, lam=lam, iters=80)
    show(ax, f.T(XG), "data only" if lam == 0 else "data + eikonal", Xd)
plt.tight_layout(); plt.show()

In [ ]:
# Checks
for N in (24, 48):
    assert np.mean([e for e, _ in rows[(N, 100)]]) < 0.5 * np.mean([e for e, _ in rows[(N, 0)]])
    assert abs(np.mean([s for _, s in rows[(N, 100)]]) - 1.0) < 0.1

## Part C · A slow region

Now a disc of radius 10 mm around (35, 30) mm conducts at 0.6 m/s; the rest at 1.0 m/s. The true map comes from
the eikonal solver of m30 on a 1-mm mesh. With 160 measurements, compare the data-only fit with a physics-informed
fit that assumes one speed everywhere. Estimate conduction velocity from each fitted map as $1/\lvert\nabla T\rvert$
(m25) inside the disc.

In [ ]:
verts, tris = mp.plane_mesh(61, 61, 1.0)
ct = verts[tris].mean(1)[:, :2]
speed = np.where(np.hypot(*(ct - [35, 30]).T) <= 10, 0.6, 1.0)
s0 = int(np.argmin(np.hypot(*(verts[:, :2] - [5, 30]).T)))
truth = LinearNDInterpolator(verts[:, :2], mp.eikonal(verts, tris, [s0], speed))
Tg = truth(XG)
inside = np.hypot(*(XG - [35, 30]).T) < 7
slow = {"data only": [], "physics, one speed": []}
for seed in range(5):
    r = np.random.default_rng(200 + seed)
    Xd = r.uniform(1, 59, (160, 2)); td = truth(Xd) + r.normal(0, 1.0, 160)
    for name, lam in (("data only", 0), ("physics, one speed", 30)):
        f = pinn.ActivationFit(CENT, H, "scalar", s_prior=1.0); f.fit(Xd, td, XC, lam=lam, iters=100)
        cv = 1 / np.linalg.norm(f.gradT(XG), axis=1)
        slow[name].append((np.sqrt(np.mean((f.T(XG) - Tg) ** 2)), np.median(cv[inside])))
for name, vals in slow.items():
    vals = np.array(vals)
    print(f"{name:19s} map RMS error {vals[:, 0].mean():.2f} ms; speed inside the slow disc {vals[:, 1].mean():.2f} m/s (true 0.6)")

In [ ]:
# Checks
d_, p_ = np.array(slow["data only"]), np.array(slow["physics, one speed"])
assert p_[:, 0].mean() < d_[:, 0].mean()                              # physics lowers map error
assert abs(d_[:, 1].mean() - 0.6) < abs(p_[:, 1].mean() - 0.6)        # but pulls the slow region's speed up

## Part D · Ensemble uncertainty

Measurements cover only the lower-left 40-by-40-mm part of the square. Fit eight physics-informed members, each to
data with fresh 1-ms noise and with a fixed random prior function added (randomized priors, as in Sahli Costabal
et al. 2020). The spread across members should be small near data, large far from it, and should track the
actual error.

In [ ]:
from scipy.stats import spearmanr

r = np.random.default_rng(5)
Xd = r.uniform(0, 40, (20, 2)); td = T_true(Xd) + r.normal(0, 1.0, 20)
preds = pinn.ensemble(Xd, td, XC, 8, lambda: pinn.ActivationFit(CENT, H, "scalar", s_prior=1.2), lam=100,
                      noise=1.0, prior_scale=3.0, rng=r, Xeval=XG, iters=40)
mean, sd = pinn.ensemble_stats(preds)
err = np.abs(mean - T_true(XG))
dist = np.min(np.linalg.norm(XG[:, None] - Xd[None], axis=2), 1)
rho = spearmanr(sd, err)[0]
cover = np.mean(err < 2 * sd)
print(f"spread near data (< 5 mm) {np.median(sd[dist < 5]):.1f} ms, far (> 15 mm) {np.median(sd[dist > 15]):.1f} ms")
print(f"rank correlation of spread with error {rho:.2f}; error within two spreads at {cover:.0%} of grid points")
fig, axs = plt.subplots(1, 2, figsize=(7, 3.2))
show(axs[0], err, "absolute error (ms)", Xd, 0, 15, "magma")
show(axs[1], sd, "ensemble spread (ms)", Xd, 0, 15, "magma")
plt.tight_layout(); plt.show()

In [ ]:
# Checks
assert np.median(sd[dist > 15]) > 3 * np.median(sd[dist < 5])
assert rho > 0.5

## Part E · A surrogate for calibration, and where it is safe

Calibrating a digital twin (m33) means running the model many times. Train a cheap surrogate of the eikonal model:
activation times at 20 measurement sites as a function of the two speeds, written in slowness ($1/v$) because
activation times along any fixed path are linear in slowness (eq-m42-scaling). Train it on 30 runs with the
outside speed in 0.8–1.2 m/s and the slow-disc speed in 0.4–0.8 m/s, calibrate against noisy measurements, and
test it inside and outside that range.

In [ ]:
v2, t2 = mp.plane_mesh(31, 31, 2.0)
disc = np.hypot(*(v2[t2].mean(1)[:, :2] - [35, 30]).T) <= 10
src2 = int(np.argmin(np.hypot(*(v2[:, :2] - [5, 30]).T)))
meas = r.choice(len(v2), 20, replace=False)
model = lambda vo, vi: mp.eikonal(v2, t2, [src2], np.where(disc, vi, vo))[meas]

P = np.c_[r.uniform(0.8, 1.2, 30), r.uniform(0.4, 0.8, 30)]
Y = np.array([model(*p) for p in P])
feats = lambda p: (lambda a, b: np.c_[np.ones(len(a)), 1 / a, 1 / b, 1 / a**2, 1 / b**2, 1 / (a * b)])(*np.atleast_2d(p).T)
Wsur = np.linalg.lstsq(feats(P), Y, rcond=None)[0]
surrogate = lambda p: feats(p) @ Wsur

checks_E = {}
for label, p in [("inside range", (1.0, 0.6)), ("inside range", (0.9, 0.5)), ("outside range", (1.4, 0.3)), ("outside range", (0.6, 0.9))]:
    e = np.sqrt(np.mean((surrogate(p)[0] - model(*p)) ** 2))
    checks_E.setdefault(label, []).append(e)
    print(f"{label:13s} speeds {p}: surrogate RMS error {e:.3f} ms")

obs = model(1.05, 0.55) + r.normal(0, 1.0, 20)
GA, GB = np.meshgrid(np.linspace(0.8, 1.2, 81), np.linspace(0.4, 0.8, 81), indexing="ij")
grid = np.c_[GA.ravel(), GB.ravel()]
best = grid[np.argmin(((surrogate(grid) - obs) ** 2).sum(1))]
print(f"calibrated speeds {best.round(3)} (true 1.05, 0.55) from {len(grid)} surrogate evaluations")

In [ ]:
# Checks
assert max(checks_E["inside range"]) < 0.2 < 1.0 < min(checks_E["outside range"])
assert abs(best[0] - 1.05) < 0.05 and abs(best[1] - 0.55) < 0.08

## Part F · Optional: a physics-informed neural network

The same homogeneous problem as part B with 24 points, solved the way Raissi et al. proposed: a small network
$T_\theta(\mathbf x)$, its gradient by automatic differentiation, and the loss of eq-m42-loss with a learned speed.
Runs only if PyTorch is installed.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part F skipped.")

if torch is not None:
    torch.manual_seed(0)
    r = np.random.default_rng(101)
    Xd = r.uniform(0, 60, (24, 2)); td = T_true(Xd) + r.normal(0, 1.0, 24)
    scale = 60.0
    xd = torch.tensor(Xd / scale, dtype=torch.float32); yd = torch.tensor(td, dtype=torch.float32)
    xc = torch.tensor(XC / scale, dtype=torch.float32, requires_grad=True)
    net = torch.nn.Sequential(torch.nn.Linear(2, 32), torch.nn.Tanh(), torch.nn.Linear(32, 32), torch.nn.Tanh(), torch.nn.Linear(32, 1))
    log_s = torch.nn.Parameter(torch.tensor(0.0))
    opt = torch.optim.Adam(list(net.parameters()) + [log_s], lr=3e-3)
    for it in range(3000):
        opt.zero_grad()
        Tc = net(xc).squeeze(1) * 60
        g = torch.autograd.grad(Tc.sum(), xc, create_graph=True)[0] / scale
        loss = ((net(xd).squeeze(1) * 60 - yd) ** 2).mean() + 100 * ((torch.exp(log_s) * g.norm(dim=1) - 1) ** 2).mean()
        loss.backward()
        opt.step()
    with torch.no_grad():
        T_nn = net(torch.tensor(XG / scale, dtype=torch.float32)).squeeze(1).numpy() * 60
    pinn_err = np.sqrt(np.mean((T_nn - T_true(XG)) ** 2))
    print(f"PINN: RMS error {pinn_err:.2f} ms, speed estimate {1 / np.exp(log_s.item()):.2f} m/s")

In [ ]:
# Checks
if torch is not None:
    assert pinn_err < 10

## Where to go next

- In part B, try $\lambda$ = 1 and 10 000. Which way does the error go, and why?
- In part C, give the physics-informed fit a speed field instead of one speed (`slowness="field"`). How many
  measurements does it need before the slow region's speed comes out right?
- In part E, widen the training range. How many training runs keep the error below 0.2 ms?